# 📘 Notebook 05 — Explainable AI (XAI) with TreeSHAP

## 🏢 STAGE 17 — Global & Individual Churn Risk Drivers
This notebook solves the black-box trade-off by using **TreeSHAP** to provide:
1. **Global Explanations**: What features drive churn across the entire customer base?
2. **Individual Explanations**: Why does Customer #152 have an 87% churn probability?

> **The XAI Value Proposition:** *SHAP provides mathematical consistency (Shapley values) to show exactly how much each feature pushes an individual customer towards churn ($\uparrow$) or anchors their loyalty ($\downarrow$).*

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.models.train_xgboost import load_model
from src.preprocessing.pipeline import ChurnPreprocessingPipeline
from src.explainability.shap_explainer import ChurnShapExplainer
from src.features.feature_builder import build_features

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

artifacts = load_model("../models/xgboost_model.pkl" if os.path.exists("../models") else "models/xgboost_model.pkl")
pipeline = ChurnPreprocessingPipeline.load("../models/preprocessor.pkl" if os.path.exists("../models") else "models/preprocessor.pkl")
base_xgb = artifacts["base_model"]
feature_names = artifacts["feature_names"]

explainer = ChurnShapExplainer(base_xgb, feature_names)
print("TreeSHAP Explainer Initialized Successfully!")

## 🌐 1. Global Explanation: Portfolio Churn Drivers

In [ ]:
DATA_PATH = Path("../data/raw") if (Path("../data/raw") / "customer_churn.csv").exists() else Path("data/raw")
df_raw = pd.read_csv(DATA_PATH / "customer_churn.csv")
df_feat = build_features(df_raw)
X_full = pipeline.transform(df_feat)

# Generate and display global SHAP plots
explainer.generate_global_plots(X_full[:500], output_dir="../figures" if os.path.exists("../figures") else "figures")

## 👤 2. Individual Customer Explanation: Case Study Customer #152

In [ ]:
# Explain Customer #152 (Row 152 in dataset)
cust_152_features = X_full[152]
res = explainer.explain_single_customer(cust_152_features, customer_id="Customer #152 (CUST-1152)")
print(res["formatted_summary"])

## 💡 3. Key XAI Conclusions

1. **Global Pattern**: Month-to-Month contracts, Support Friction, and Low CSAT are the top 3 global drivers of customer churn across the business.
2. **Individual Actionability**: Customer #152 is churning due to **Support Call Friction (5 calls)** + **Month-to-Month Contract**. Local SHAP drivers allow customer support to offer targeted resolution instead of generic discount codes.

---